In [ ]:
"""
Problem 7 - Stochastic gridworld / parking problem (EE 5531 Assignment-2).

Environment read off Figure 2 (4x4 grid, row 0 at the top):

    (0,0) 0        (0,1) 0        (0,2) 0    (0,3) +10 GOAL
    (1,0) 0        (1,1) OBSTACLE (1,2) 0    (1,3) -50 PIT
    (2,0) 0        (2,1) 0        (2,2) 0    (2,3) 0
    (3,0) -50 PIT  (3,1) OBSTACLE (3,2) 0    (3,3) 0

Actions move in the intended direction with probability 0.8, and in each of
the two perpendicular directions with probability 0.1 (Figure 3). Bumping a
wall or an obstacle leaves the agent where it is. gamma = 0.9.
"""

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

In [ ]:
GAMMA = 0.9
N = 4

In [ ]:
GOAL = (0, 3)
PITS = {(1, 3), (3, 0)}
OBSTACLES = {(1, 1), (3, 1)}
TERMINALS = {GOAL} | PITS

In [ ]:
ACTIONS = {"U": (-1, 0), "D": (1, 0), "L": (0, -1), "R": (0, 1)}
PERP = {"U": ("L", "R"), "D": ("L", "R"), "L": ("U", "D"), "R": ("U", "D")}

In [ ]:
FREE = [(r, c) for r in range(N) for c in range(N)
        if (r, c) not in OBSTACLES and (r, c) not in TERMINALS]

In [ ]:
def transitions(row, col, action):
    """{(next_row, next_col): probability} for taking `action` in (row, col).

    Outcomes are accumulated into a dict because several of the three
    branches can land on the same cell (e.g. two of them bump the same wall).
    """
    outcomes = {}
    branches = [(0.8, action), (0.1, PERP[action][0]), (0.1, PERP[action][1])]
    for prob, act in branches:
        dr, dc = ACTIONS[act]
        next_row, next_col = row + dr, col + dc
        off_grid = not (0 <= next_row < N and 0 <= next_col < N)
        if off_grid or (next_row, next_col) in OBSTACLES:
            next_row, next_col = row, col          # bump -> stay put
        outcomes[(next_row, next_col)] = outcomes.get((next_row, next_col), 0.0) + prob
    return outcomes

In [ ]:
def reward(next_row, next_col):
    """Reward for arriving in a cell: +10 at the goal, -50 at a pit, else 0."""
    if (next_row, next_col) == GOAL:
        return 10.0
    if (next_row, next_col) in PITS:
        return -50.0
    return 0.0

In [ ]:
def q_value(values, row, col, action, gamma=GAMMA):
    total = 0.0
    for (next_row, next_col), prob in transitions(row, col, action).items():
        total += prob * (reward(next_row, next_col) + gamma * values[next_row, next_col])
    return total

In [ ]:
def value_iteration(n_sweeps=10, gamma=GAMMA):
    values = np.zeros((N, N))
    history = [values.copy()]
    for _ in range(n_sweeps):
        new_values = np.zeros((N, N))
        for (row, col) in FREE:
            new_values[row, col] = max(q_value(values, row, col, a, gamma)
                                       for a in ACTIONS)
        values = new_values
        history.append(values.copy())
    return values, history

In [ ]:
def greedy_policy(values, gamma=GAMMA):
    policy = {}
    for (row, col) in FREE:
        policy[(row, col)] = max(ACTIONS,
                                 key=lambda a: q_value(values, row, col, a, gamma))
    return policy

In [ ]:
def evaluate(policy, gamma=GAMMA, theta=1e-10, max_sweeps=100000):
    """Policy evaluation for a deterministic policy, values starting at 0."""
    values = np.zeros((N, N))
    for sweep in range(1, max_sweeps + 1):
        new_values = np.zeros((N, N))
        for (row, col) in FREE:
            new_values[row, col] = q_value(values, row, col, policy[(row, col)], gamma)
        delta = np.abs(new_values - values).max()
        values = new_values
        if delta < theta:
            return values, sweep
    raise RuntimeError("policy evaluation did not converge")

In [ ]:
def show(values, title, policy=None):
    print(title)
    for row in range(N):
        cells = []
        for col in range(N):
            if (row, col) in OBSTACLES:
                cells.append("   ####")
            elif (row, col) == GOAL:
                cells.append("  GOAL ")
            elif (row, col) in PITS:
                cells.append("   PIT ")
            else:
                label = f"{values[row, col]:7.3f}"
                if policy is not None:
                    label = f"{values[row, col]:6.2f}{policy[(row, col)]}"
                cells.append(label)
        print("  " + " ".join(cells))
    print()

In [ ]:
def draw(ax, values, policy, title, show_values=True):
    ax.set_title(title, fontsize=10, pad=8)
    for row in range(N):
        for col in range(N):
            y = N - 1 - row                      # flip so row 0 draws at the top
            if (row, col) in OBSTACLES:
                face, text, label = "#1a1a1a", "white", "WALL"
            elif (row, col) == GOAL:
                face, text, label = "#1b7f3b", "white", "GOAL\n+10"
            elif (row, col) in PITS:
                face, text, label = "#c0272d", "white", "PIT\n-50"
            else:
                face, text, label = "#ffffff", "#222222", ""

            ax.add_patch(Rectangle((col, y), 1, 1, facecolor=face,
                                   edgecolor="#888888", linewidth=1.0))
            if label:
                ax.text(col + 0.5, y + 0.5, label, ha="center", va="center",
                        color=text, fontsize=8, fontweight="bold")
            else:
                if show_values:
                    ax.text(col + 0.5, y + 0.72, f"{values[row, col]:.2f}",
                            ha="center", va="center", color=text, fontsize=8)
                if policy is not None and (row, col) in policy:
                    dr, dc = ACTIONS[policy[(row, col)]]
                    ax.arrow(col + 0.5, y + 0.36, 0.26 * dc, -0.26 * dr,
                             head_width=0.11, head_length=0.09,
                             fc="#1f4e9c", ec="#1f4e9c", length_includes_head=True)

    ax.set_xlim(0, N)
    ax.set_ylim(0, N)
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

In [ ]:
if __name__ == "__main__":
    print(f"gamma = {GAMMA}, goal {GOAL} +10, pits {sorted(PITS)} -50, "
          f"obstacles {sorted(OBSTACLES)}\n")

    # Sanity: every action's transition probabilities must sum to 1
    for (row, col) in FREE:
        for a in ACTIONS:
            total = sum(transitions(row, col, a).values())
            assert abs(total - 1.0) < 1e-12, (row, col, a, total)
    print("checked: all transition distributions sum to 1\n")

    # ---- (i) value iteration, 10 iterations -------------------------------
    values10, history = value_iteration(10)
    for k in (1, 2, 3):
        show(history[k], f"(i) after {k} iteration(s):")
    show(values10, "(i) values after 10 iterations:")

    # ---- (ii) policy at the end of the 10th iteration ---------------------
    policy10 = greedy_policy(values10)
    show(values10, "(ii) greedy policy after 10 iterations:", policy10)

    converged, _ = value_iteration(200)
    print(f"for reference, max |v_10 - v_converged| = "
          f"{np.abs(values10 - converged).max():.4f}")
    print(f"policy after 10 == policy at convergence: "
          f"{greedy_policy(converged) == policy10}\n")

    # ---- (iii) policy iteration from pi_0 = Up ----------------------------
    pi0 = {s: "U" for s in FREE}
    v_pi0, n0 = evaluate(pi0)
    show(v_pi0, f"(iii) V_pi0 for pi_0 = Up everywhere ({n0} sweeps):", pi0)

    pi1 = greedy_policy(v_pi0)
    show(v_pi0, "(iii) pi_1 = greedy(V_pi0):", pi1)

    v_pi1, n1 = evaluate(pi1)
    show(v_pi1, f"(iii) V_pi1 ({n1} sweeps):", pi1)

    pi2 = greedy_policy(v_pi1)
    show(v_pi1, "(iii) pi_2 = greedy(V_pi1):", pi2)

    changed = [s for s in FREE if pi1[s] != pi2[s]]
    print(f"states where pi_2 differs from pi_1: {changed or 'none'}")
    print(f"pi_2 == optimal policy: {pi2 == greedy_policy(converged)}\n")

    # ---- figures ----------------------------------------------------------
    fig, ax = plt.subplots(figsize=(4.2, 4.4))
    draw(ax, values10, policy10, "Problem 7 (i)-(ii): values and policy\nafter 10 value-iteration sweeps")
    fig.tight_layout()
    fig.savefig("p7_value_iteration.png", dpi=150)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4.8))
    draw(axes[0], v_pi0, pi0, "$\\pi_0$ = Up everywhere,\nwith $V_{\\pi_0}$")
    draw(axes[1], v_pi0, pi1, "$\\pi_1$ = greedy($V_{\\pi_0}$),\nshown on $V_{\\pi_0}$")
    draw(axes[2], v_pi1, pi2, "$\\pi_2$ = greedy($V_{\\pi_1}$),\nshown on $V_{\\pi_1}$")
    fig.subplots_adjust(top=0.82)
    fig.savefig("p7_policy_iteration.png", dpi=150, bbox_inches="tight")
    print("figures written: p7_value_iteration.png, p7_policy_iteration.png")